In [1]:
import os
import csv
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

In [3]:
# -----------------------
# CONFIG
# -----------------------
MODEL_PATHS = [
    "cnn_model_0.h5",
    "cnn_model_1.h5",
    "cnn_model_2.h5",
    "cnn_model_3.h5",
    "cnn_model_4.h5",
]

MODEL_THRESHOLDS = np.array([0.65, 0.35, 0.60, 0.50, 0.45])

WEIGHTS = np.array([1.0, 1.0, 1.2, 1.0, 1.5])
WEIGHTS = WEIGHTS / WEIGHTS.sum()

ENSEMBLE_THRESHOLD = 0.45  # can tune this

DATASET_ROOT = "./dataset_new"
OUTPUT_CSV = "ensemble_predictions.csv"
OUTPUT_XLSX = "ensemble_predictions.xlsx"
IMG_SIZE = (224, 224)

# Excel styles
GREEN_FILL = PatternFill("solid", fgColor="C6EFCE")
RED_FILL = PatternFill("solid", fgColor="FFC7CE")
HEADER_FILL = PatternFill("solid", fgColor="BDD7EE")


In [4]:
# -----------------------
# HELPERS
# -----------------------
def preprocess(path):
    img = load_img(path, target_size=IMG_SIZE)
    arr = img_to_array(img) / 255.0
    return np.expand_dims(arr, axis=0)

def calibrate_probability(prob, threshold):
    """
    Scale probability relative to model's best threshold
    """
    if prob >= threshold:
        return (prob - threshold) / (1 - threshold)
    else:
        return prob / threshold

def ensemble_predict(models, img):
    calibrated_probs = []

    for model, thr in zip(models, MODEL_THRESHOLDS):
        p = model.predict(img, verbose=0)
        p = float(p[0][0]) if p.shape[-1] == 1 else float(p[0][1])

        calibrated_p = calibrate_probability(p, thr)
        calibrated_probs.append(calibrated_p)

    calibrated_probs = np.array(calibrated_probs)

    # Weighted soft voting
    ensemble_prob = np.sum(calibrated_probs * WEIGHTS)
    return float(ensemble_prob)


In [5]:
# -----------------------
# MAIN
# -----------------------
def main():
    print("🔄 Loading models...")
    models = [load_model(p, compile=False) for p in MODEL_PATHS]

    images = []
    for cls in ["normal", "osteoporosis"]:
        folder = os.path.join(DATASET_ROOT, cls)
        for f in os.listdir(folder):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append((os.path.join(folder, f), cls))

    wb = Workbook()
    ws = wb.active
    ws.title = "Ensemble Results"

    header = ["Image", "True Class", "Ensemble Prob", "Prediction", "Status"]
    ws.append(header)

    for i in range(1, len(header) + 1):
        ws.cell(row=1, column=i).fill = HEADER_FILL
        ws.column_dimensions[get_column_letter(i)].width = 28

    csv_rows = [header]
    correct = 0

    for img_path, true_cls in images:
        img = preprocess(img_path)
        prob = ensemble_predict(models, img)

        pred = "osteoporosis" if prob >= ENSEMBLE_THRESHOLD else "normal"
        status = "Correct" if pred == true_cls else "Wrong"
        correct += (status == "Correct")

        ws.append([os.path.basename(img_path), true_cls, round(prob, 4), pred, status])
        fill = GREEN_FILL if status == "Correct" else RED_FILL
        for c in range(1, 6):
            ws.cell(row=ws.max_row, column=c).fill = fill

        csv_rows.append([img_path, true_cls, prob, pred, status])

    with open(OUTPUT_CSV, "w", newline="") as f:
        csv.writer(f).writerows(csv_rows)

    wb.save(OUTPUT_XLSX)

    acc = correct / len(images) * 100
    print(f"\n✅ Ensemble Accuracy: {acc:.2f}%")
    print("📁 Saved CSV & XLSX")

if __name__ == "__main__":
    main()

🔄 Loading models...

✅ Ensemble Accuracy: 89.67%
📁 Saved CSV & XLSX
